# 00 — Learn the code (practice notebook)

Notebooks 01 and 02 use about ten pandas tools that basic courses often skip. This notebook teaches each one on a **tiny made-up table**, so you can see every row change.

**How to use it**
1. Run one cell at a time (Shift + Enter).
2. Before running a cell, **guess** what it will print. Then run it and compare.
3. Do the "Try it" exercise, then open the answer.
4. Say out loud, in one sentence, what the line does. That is what interviews test.

| Section | Tool | Used in |
|---|---|---|
| 1 | `isin` with a set | NB01 step 2 |
| 2 | `groupby().agg(name=(col, func))` | NB01 steps 3, 5 · NB02 step 2 |
| 3 | `transform("max")` | NB01 step 5b · NB02 step 3 |
| 4 | Finding the Monday of a week | NB02 step 2 |
| 5 | `pd.cut` for age bands | NB02 step 1 |
| 6 | `map` as a lookup | NB02 step 2 |
| 7 | `join` / `merge` | NB01 step 6 · NB02 step 2 |
| 8 | Reading a file in chunks | NB01 step 2 · NB02 step 2 |
| 9 | `unstack` | NB02 step 4 |
| 10 | `cumsum` and "first week reaching 50%" | NB02 step 4 |

Everything here is made-up data; it does not touch the real files.

## 0. The tiny tables we will use

In [1]:
import io
import numpy as np
import pandas as pd

# 10 purchases, same columns as transactions_train.csv
tx = pd.DataFrame({
    "t_dat": pd.to_datetime(["2019-07-01", "2019-07-03", "2019-07-03", "2019-07-06", "2019-07-09",
                             "2019-07-12", "2019-07-02", "2019-07-10", "2019-07-10", "2019-07-11"]),
    "customer_id": ["A", "B", "C", "D", "E", "F", "A", "A", "B", "G"],
    "article_id": [101, 101, 101, 101, 101, 101, 202, 202, 202, 202],
    "price": [0.030, 0.030, 0.030, 0.020, 0.020, 0.020, 0.050, 0.050, 0.025, 0.050],
    "sales_channel_id": [2, 2, 2, 2, 2, 2, 1, 1, 1, 2],
})

# 7 customers with their ages (G has no age)
customers = pd.DataFrame({
    "customer_id": ["A", "B", "C", "D", "E", "F", "G"],
    "age": [19, 23, 31, 47, 52, 60, np.nan],
})

# 2 articles with their product info
articles = pd.DataFrame({
    "article_id": [101, 202],
    "product_type_name": ["T-shirt", "Dress"],
    "garment_group_name": ["Jersey Basic", "Dresses Ladies"],
})

tx

,t_dat,customer_id,article_id,price,sales_channel_id
0,2019-07-01,A,101,0.030,2
1,2019-07-03,B,101,0.030,2
2,2019-07-03,C,101,0.030,2
3,2019-07-06,D,101,0.020,2
4,2019-07-09,E,101,0.020,2
5,2019-07-12,F,101,0.020,2
6,2019-07-02,A,202,0.050,1
7,2019-07-10,A,202,0.050,1
8,2019-07-10,B,202,0.025,1
9,2019-07-11,G,202,0.050,2


## 1. `isin` with a set: keep rows for chosen customers

**Used in NB01 step 2.** We picked 10% of customers and want only their purchases.

`isin` answers True/False for each row: "is this value in my list?". Putting that True/False column inside `tx[...]` keeps only the True rows.

In [2]:
sample_ids = {"A", "C", "E"}          # a Python set: checking membership is very fast

tx["customer_id"].isin(sample_ids)    # step 1: True/False for every row

0     True
1    False
2     True
3    False
4     True
5    False
6     True
7     True
8    False
9    False
Name: customer_id, dtype: bool

In [3]:
tx[tx["customer_id"].isin(sample_ids)]   # step 2: keep only the True rows

,t_dat,customer_id,article_id,price,sales_channel_id
0,2019-07-01,A,101,0.03,2
2,2019-07-03,C,101,0.03,2
4,2019-07-09,E,101,0.02,2
6,2019-07-02,A,202,0.05,1
7,2019-07-10,A,202,0.05,1


**Why a set and not a list?** Asking "is X in this set?" is instant even with 137,000 IDs. With a list, Python checks items one by one. On 31.8M rows that difference is minutes.

**`~` means NOT.** NB01 step 4 uses `~tx["article_id"].isin(...)` to find rows whose article is *missing* from the articles table.

**Try it:** keep the purchases of customers who are **not** A.

In [ ]:
~tx["article_id"].isin({A"})   # step 1: True/False for every row

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
tx[~tx["customer_id"].isin({"A"})]
```
</details>

## 2. `groupby().agg(new_name=(column, how))`: several summaries at once

**Used everywhere.** Read it as: "for each group, make these new columns".

Each line inside `agg` is `new_column_name=(column_to_use, what_to_do)`.

In [5]:
tx.groupby("article_id").agg(
    units=("price", "size"),        # how many rows in the group
    sales_value=("price", "sum"),   # add the prices
    min_price=("price", "min"),
    max_price=("price", "max"),
)

,units,sales_value,min_price,max_price
article_id,,,,
101,6,0.150,0.020,0.03
202,4,0.175,0.025,0.05


`size` counts rows, so it doesn't matter which column you give it; the notebooks use `"price"`.

**Grouping by two columns** gives one row per combination. This is how NB01 builds **baskets** (one customer on one date):

In [6]:
baskets = tx.groupby(["customer_id", "t_dat"]).agg(
    items=("article_id", "size"), value=("price", "sum")
).reset_index()      # reset_index turns the group labels back into normal columns
baskets

,customer_id,t_dat,items,value
0,A,2019-07-01,1,0.030
1,A,2019-07-02,1,0.050
2,A,2019-07-10,1,0.050
3,B,2019-07-03,1,0.030
4,B,2019-07-10,1,0.025
5,C,2019-07-03,1,0.030
6,D,2019-07-06,1,0.020
7,E,2019-07-09,1,0.020
8,F,2019-07-12,1,0.020
9,G,2019-07-11,1,0.050


Check it: customer A shopped on 07-01, 07-02 and 07-10, so A has 3 baskets. Customer B shopped on 07-03 and 07-10, so B has 2.

**Try it:** for each `sales_channel_id`, count the rows and the number of **different** customers (`"nunique"`).

In [7]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
tx.groupby("sales_channel_id").agg(
    rows=("price", "size"),
    customers=("customer_id", "nunique"),
)
```
</details>

## 3. `transform("max")`: put the group's answer back on every row

**Used in NB01 step 5b** (discount check) and **NB02 step 3** (share of peak).

A normal `groupby().max()` gives **one row per group**. `transform("max")` gives **one value per original row**: each row receives its own group's max. That lets us compare each purchase with its article's highest price.

In [8]:
# Normal groupby: one row per article/channel
tx.groupby(["article_id", "sales_channel_id"])["price"].max()

article_id  sales_channel_id
101         2                   0.03
202         1                   0.05
            2                   0.05
Name: price, dtype: float64

In [9]:
# transform: same answer, but copied onto every row, so it lines up with tx
tx["ref_price"] = tx.groupby(["article_id", "sales_channel_id"])["price"].transform("max")
tx["price_ratio"] = tx["price"] / tx["ref_price"]
tx["markdown"] = tx["price_ratio"] <= 0.85
tx[["article_id", "sales_channel_id", "price", "ref_price", "price_ratio", "markdown"]]

,article_id,sales_channel_id,price,ref_price,price_ratio,markdown
0,101,2,0.030,0.03,1.000000,False
1,101,2,0.030,0.03,1.000000,False
2,101,2,0.030,0.03,1.000000,False
3,101,2,0.020,0.03,0.666667,True
4,101,2,0.020,0.03,0.666667,True
5,101,2,0.020,0.03,0.666667,True
6,202,1,0.050,0.05,1.000000,False
7,202,1,0.050,0.05,1.000000,False
8,202,1,0.025,0.05,0.500000,True
9,202,2,0.050,0.05,1.000000,False


Look at article 202 in channel 1: the max is 0.050, so the 0.025 sale has ratio 0.5 → markdown. Article 202 in channel 2 is compared only with channel 2 prices.

**The mean of True/False is a share.** True counts as 1 and False as 0, so `.mean()` gives the percentage of markdown purchases. NB01 uses this trick a lot.

In [10]:
print(f"Markdown share: {tx['markdown'].mean():.0%}")

Markdown share: 40%


**Try it:** add a column `share_of_article_units` = 1 divided by the number of rows for that article (hint: `transform("size")`).

In [11]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
tx["share_of_article_units"] = 1 / tx.groupby("article_id")["price"].transform("size")
tx[["article_id", "share_of_article_units"]]
```
</details>

## 4. Finding the Monday of each week

**Used in NB02 step 2.** We want every date labelled with the Monday that starts its week.

- `dt.dayofweek` gives 0 for Monday, 1 for Tuesday … 6 for Sunday.
- Going back that many days always lands on Monday.

In [12]:
weeks = tx[["t_dat"]].copy()
weeks["day_name"] = weeks["t_dat"].dt.day_name()
weeks["dayofweek"] = weeks["t_dat"].dt.dayofweek
weeks["week"] = weeks["t_dat"] - pd.to_timedelta(weeks["t_dat"].dt.dayofweek, unit="D")
weeks

,t_dat,day_name,dayofweek,week
0,2019-07-01,Monday,0,2019-07-01
1,2019-07-03,Wednesday,2,2019-07-01
2,2019-07-03,Wednesday,2,2019-07-01
3,2019-07-06,Saturday,5,2019-07-01
4,2019-07-09,Tuesday,1,2019-07-08
5,2019-07-12,Friday,4,2019-07-08
6,2019-07-02,Tuesday,1,2019-07-01
7,2019-07-10,Wednesday,2,2019-07-08
8,2019-07-10,Wednesday,2,2019-07-08
9,2019-07-11,Thursday,3,2019-07-08


`pd.to_timedelta(number, unit="D")` turns a number into "that many days", which can be subtracted from a date.

**Try it:** group `tx` by the new week and count the units sold per article per week. You should get 4 rows.

In [13]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
tx["week"] = tx["t_dat"] - pd.to_timedelta(tx["t_dat"].dt.dayofweek, unit="D")
tx.groupby(["article_id", "week"]).agg(units=("price", "size")).reset_index()
```
</details>

## 5. `pd.cut`: put ages into bands

**Used in NB02 step 1.** `bins` are the edges. Each band covers *above the left edge, up to and including the right edge*: (24, 34] means 25 to 34.

In [14]:
AGE_BINS = [0, 24, 34, 44, 54, 200]
AGE_LABELS = ["16-24", "25-34", "35-44", "45-54", "55+"]

bands = customers.copy()
bands["band_text"] = pd.cut(bands["age"], bins=AGE_BINS, labels=AGE_LABELS)
bands["band_number"] = pd.cut(bands["age"], bins=AGE_BINS, labels=False)
bands

,customer_id,age,band_text,band_number
0,A,19.0,16-24,0.0
1,B,23.0,16-24,0.0
2,C,31.0,25-34,1.0
3,D,47.0,45-54,3.0
4,E,52.0,45-54,3.0
5,F,60.0,55+,4.0
6,G,NaN,NaN,NaN


NB02 uses `labels=False`, which gives numbers 0–4 instead of text: smaller and faster. G has no age, so its band is missing (NaN). NB02 fills that with -1 = "unknown":

In [15]:
bands["age_band"] = bands["band_number"].fillna(-1).astype("int8")
bands

,customer_id,age,band_text,band_number,age_band
0,A,19.0,16-24,0.0,0
1,B,23.0,16-24,0.0,0
2,C,31.0,25-34,1.0,1
3,D,47.0,45-54,3.0,3
4,E,52.0,45-54,3.0,3
5,F,60.0,55+,4.0,4
6,G,NaN,NaN,NaN,-1


**Try it:** make a column `is_young` that is True when the age band number is 0.

In [16]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
bands["is_young"] = bands["age_band"].eq(0)
bands
```
</details>

## 6. `map` with a Series: a lookup, like VLOOKUP

**Used in NB02 step 2.** Each purchase needs its buyer's age band.

1. `set_index("customer_id")` turns the customers table into a lookup: give it an ID, get back a value.
2. `tx["customer_id"].map(lookup)` looks up every row.

In [17]:
age_band_of = bands.set_index("customer_id")["age_band"]
age_band_of        # the lookup: ID -> age band

customer_id
A    0
B    0
C    1
D    3
E    3
F    4
G   -1
Name: age_band, dtype: int8

In [18]:
tx["age_band"] = tx["customer_id"].map(age_band_of)
tx[["customer_id", "age_band"]]

,customer_id,age_band
0,A,0
1,B,0
2,C,1
3,D,3
4,E,3
5,F,4
6,A,0
7,A,0
8,B,0
9,G,-1


A customer missing from the lookup would get NaN. That's why NB02 adds `.fillna(-1)` after `map`.

**Try it:** build a lookup from `articles` and add a `product_type_name` column to `tx` with `map`.

In [19]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
type_of = articles.set_index("article_id")["product_type_name"]
tx["product_type_name"] = tx["article_id"].map(type_of)
tx[["article_id", "product_type_name"]]
```
</details>

## 7. `join` and `merge`: SQL JOIN in pandas

**Used in NB01 step 6 and NB02 step 2.** `map` adds one column; `join`/`merge` can add several at once.

- `a.merge(b, on="key")` matches rows on a column both tables have.
- `a.join(b, on="key")` does the same, but `b` must have the key as its **index**.

NB02 uses `join` because `attrs` already has `article_id` as its index.

In [20]:
attrs = articles.set_index("article_id")[["garment_group_name"]]
tx.join(attrs, on="article_id")[["article_id", "garment_group_name"]]

,article_id,garment_group_name
0,101,Jersey Basic
1,101,Jersey Basic
2,101,Jersey Basic
3,101,Jersey Basic
4,101,Jersey Basic
5,101,Jersey Basic
6,202,Dresses Ladies
7,202,Dresses Ladies
8,202,Dresses Ladies
9,202,Dresses Ladies


In [21]:
# The same with merge (no index needed). Row count must not change:
merged = tx.merge(articles[["article_id", "garment_group_name"]], on="article_id")
assert len(merged) == len(tx), "Row count changed during merge"
print(len(tx), "rows before,", len(merged), "rows after")

10 rows before, 10 rows after


**`assert` is a safety check.** If the condition is False, the notebook stops with your message. Try changing `==` to `!=` above and run it to see what an assert failure looks like, then change it back.

## 8. Reading a file in chunks

**Used in NB01 step 2 and NB02 step 2.** The real file has 31.8M rows; we read it 2M at a time. Here we pretend: a 10-row "file" read **4 rows at a time**.

With `chunksize=`, `read_csv` doesn't return a table. It returns a **reader** that hands out one piece each time the `for` loop asks.

In [22]:
csv_text = tx[["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]].to_csv(index=False)
fake_file = io.StringIO(csv_text)     # behaves like a file on disk

reader = pd.read_csv(fake_file, chunksize=4, parse_dates=["t_dat"])
for i, chunk in enumerate(reader, 1):
    print(f"--- chunk {i}: {len(chunk)} rows ---")
    print(chunk)

--- chunk 1: 4 rows ---
       t_dat customer_id  article_id  price  sales_channel_id
0 2019-07-01           A         101   0.03                 2
1 2019-07-03           B         101   0.03                 2
2 2019-07-03           C         101   0.03                 2
3 2019-07-06           D         101   0.02                 2
--- chunk 2: 4 rows ---
       t_dat customer_id  article_id  price  sales_channel_id
4 2019-07-09           E         101   0.02                 2
5 2019-07-12           F         101   0.02                 2
6 2019-07-02           A         202   0.05                 1
7 2019-07-10           A         202   0.05                 1
--- chunk 3: 2 rows ---
       t_dat customer_id  article_id  price  sales_channel_id
8 2019-07-10           B         202  0.025                 1
9 2019-07-11           G         202  0.050                 2


**The NB01 pattern: filter each chunk, keep what you need, glue at the end.**

In [23]:
reader = pd.read_csv(io.StringIO(csv_text), chunksize=4, parse_dates=["t_dat"])
kept = []
for chunk in reader:
    kept.append(chunk[chunk["customer_id"].isin(sample_ids)])   # keep only sampled customers
sample = pd.concat(kept, ignore_index=True)                      # glue the kept pieces
sample

,t_dat,customer_id,article_id,price,sales_channel_id
0,2019-07-01,A,101,0.03,2
1,2019-07-03,C,101,0.03,2
2,2019-07-09,E,101,0.02,2
3,2019-07-02,A,202,0.05,1
4,2019-07-10,A,202,0.05,1


**The NB02 pattern: summarise each chunk, then summarise the summaries.**

A group can be split across chunks (article 101 appears in chunks 1 and 2). So after the loop we group **again** and add the pieces. The assert checks no row was lost.

In [24]:
reader = pd.read_csv(io.StringIO(csv_text), chunksize=4, parse_dates=["t_dat"])
parts, total = [], 0
for chunk in reader:
    total += len(chunk)
    parts.append(chunk.groupby("article_id").agg(units=("price", "size")).reset_index())

print("Pieces before combining:")
print(pd.concat(parts))

combined = pd.concat(parts).groupby("article_id").agg(units=("units", "sum")).reset_index()
assert combined["units"].sum() == total, "Units don't add up to total rows"
print("\nAfter combining:")
print(combined)

Pieces before combining:
   article_id  units
0         101      4
0         101      2
1         202      2
0         202      2

After combining:
   article_id  units
0         101      6
1         202      4


Note: when combining, `units` are **summed** (not counted again), mins take the **min of the mins**, and maxes take the **max of the maxes**. That's exactly what NB02's final `groupby` does.

## 9. `unstack`: long table → grid

**Used in NB02 step 4.** A groupby with two keys gives a "long" table. `unstack` moves the last key into columns, making a grid: one row per product, one column per week.

In [25]:
tx["week"] = tx["t_dat"] - pd.to_timedelta(tx["t_dat"].dt.dayofweek, unit="D")
tx["product_type_name"] = tx["article_id"].map(articles.set_index("article_id")["product_type_name"])
long =tx.groupby(["product_type_name", "week"])["price"].size()
long

product_type_name  week      
Dress              2019-07-01    1
                   2019-07-08    3
T-shirt            2019-07-01    4
                   2019-07-08    2
Name: price, dtype: int64

In [26]:
grid = long.unstack(fill_value=0)    # fill_value=0: a week with no sales shows 0, not NaN
grid

week,2019-07-01,2019-07-08
product_type_name,,
Dress,1,3
T-shirt,4,2


`pivot_table` (used in NB02 step 3) does the same kind of reshaping in one step, and can average values while it does.

**Try it:** make a grid with one row per `customer_id` and one column per `article_id`, showing how many items each customer bought.

In [27]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
tx.groupby(["customer_id", "article_id"])["price"].size().unstack(fill_value=0)
```
</details>

## 10. `cumsum`: running total, and "first week reaching 50%"

**Used in NB02 step 4** to find when each age band had made half its purchases of a product type.

In [28]:
weekly = pd.Series([2, 6, 8, 3, 1],
                   index=pd.to_datetime(["2019-07-01", "2019-07-08", "2019-07-15",
                                         "2019-07-22", "2019-07-29"]),
                   name="units")
running = weekly.cumsum()            # 2, 8, 16, 19, 20
share = running / weekly.sum()       # running total as a share of the total
pd.DataFrame({"units": weekly, "running_total": running, "share_so_far": share.round(2)})

,units,running_total,share_so_far
2019-07-01,2,2,0.10
2019-07-08,6,8,0.40
2019-07-15,8,16,0.80
2019-07-22,3,19,0.95
2019-07-29,1,20,1.00


In [29]:
# Keep the weeks where the share is at least 50%, and take the first one
week_50 = share[share >= 0.5].index[0]
print("Reached 50% of its purchases in the week of", week_50.date())

Reached 50% of its purchases in the week of 2019-07-15


If the 16–24 band reaches 50% in an earlier week than the 45–54 band, the younger group bought that product earlier.

**Try it:** find the first week where the share reaches **90%**.

In [30]:
# your code here

<details><summary><b>Show answer</b> (try it yourself first!)</summary>

```python
share[share >= 0.9].index[0].date()
```
</details>

## What next

Open notebook 01 or 02 and read one step at a time. Every line should now match one of the ten tools above. If a line still doesn't make sense, copy it into a new cell here and try it on the tiny tables.